In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def build_churn_features(sessions: pd.DataFrame) -> dict:
    # ── Input validation ──────────────────────────────────
    assert 'user_id'      in sessions.columns, "missing 'user_id'"
    assert 'duration_min' in sessions.columns, "missing 'duration_min'"
    assert 'churned'      in sessions.columns, "missing 'churned'"

    # ── Step 1: Aggregate to one row per user ─────────────
    user_df = sessions.groupby('user_id').agg(
        total_sessions = ('duration_min', 'count'),
        avg_duration   = ('duration_min', 'mean'),
        total_minutes  = ('duration_min', 'sum'),
        churned        = ('churned',      'first')
    ).reset_index()

    # ── Step 2: Extract X and y ───────────────────────────
    feature_cols = ['total_sessions', 'avg_duration', 'total_minutes']
    X = user_df[feature_cols].values
    y = user_df['churned'].values          # single bracket → 1D array

    # ── Step 3: Standardize features ─────────────────────
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ── Step 4: Fit logistic regression ──────────────────
    model = LogisticRegression(max_iter=1000)
    model.fit(X_scaled, y)

    # ── Step 5: Feature ranking ───────────────────────────
    coefficients    = model.coef_[0]       # numpy array, shape (3,)
    feature_ranking = sorted(
        [(name, abs(coef)) for name, coef in zip(feature_cols, coefficients)],
        key=lambda x: x[1],
        reverse=True
    )
    top_churn_driver = feature_ranking[0][0]

    # ── Step 6: AUC ───────────────────────────────────────
    auc = roc_auc_score(y, model.predict_proba(X_scaled)[:, 1])

    # ── Output validation ─────────────────────────────────
    assert 0.5 <= auc <= 1.0,                              "AUC should be > 0.5"
    assert len(feature_ranking) == len(feature_cols),      "ranking length mismatch"
    assert top_churn_driver in feature_cols,               "top driver must be valid feature"
    assert feature_ranking[0][0] == top_churn_driver,      "top driver must match ranking[0]"

    return {
        'top_churn_driver': top_churn_driver,
        'auc'             : round(auc, 4),
        'n_users'         : len(user_df),
        'feature_ranking' : feature_ranking
    }


# ── Test ──────────────────────────────────────────────────
np.random.seed(1)
n_users  = 300
user_ids = [f'u{i}' for i in range(n_users)]

rows = []
for uid in user_ids:
    n_sessions = np.random.randint(1, 20)
    churned    = int(np.random.rand() < 0.3)
    for _ in range(n_sessions):
        rows.append({
            'user_id'     : uid,
            'session_date': '2025-09-01',
            'duration_min': np.random.exponential(20),
            'churned'     : churned
        })

sessions = pd.DataFrame(rows)
result   = build_churn_features(sessions)

assert 0.5 <= result['auc'] <= 1.0
assert result['n_users'] == n_users
assert result['top_churn_driver'] in ['total_sessions', 'avg_duration', 'total_minutes']
assert result['feature_ranking'][0][0] == result['top_churn_driver']
print("All tests passed ✅")
print(result)